# Using resources (Files)

Notebook version of [`using-resources.py`](./using-resources.py) — working with Files (resources) on the Istari Digital Platform.

Uses **istari-digital-client 10.10.0** — `V3Client` (V3 resources API) where available, otherwise the v2 `Client`.

Reads `ISTARI_REGISTRY_URL` and `ISTARI_PERSONAL_ACCESS_TOKEN` from [`samples/.env`](../.env).

### Actions demonstrated

- Upload a resource from your device (with and without `display_name`)
- Upload with external identifier and version label
- Search, filter, and get resources by id, revision, and external identifiers; resolve current user for owner filter; browse with cursor pagination
- Add a comment on a resource
- View resource details, versions, comments, and the list of users the resource is shared with (and their roles)
- Upload a new version, compare revisions, download content
- Archive a throwaway resource (main demo resources stay active for UI review)
- Cleanup: archive every resource created in this run (final cell)

### 1 · Install dependencies

From the cookbook repository root:

```bash
uv sync --group dev
```

### 2 · Register the venv as a Jupyter kernel

```bash
uv run python -m ipykernel install --user --name istari-client-cookbook --display-name "Python (istari-client-cookbook)"
```

Open this notebook in VS Code or Cursor and select **Python (istari-client-cookbook)** in the kernel picker.

To remove the kernel later: `jupyter kernelspec uninstall istari-client-cookbook`.

### Running cells individually

Every demo cell can be run on its own after the **Setup** cell (cell 2). Each cell has a `── Standalone usage ──` comment at the top explaining which inputs (typically `resource_id`) to set. When the prior cells have already populated those variables, the cell reuses them; otherwise it falls back to a `REPLACE_WITH_RESOURCE_ID` placeholder you must override (or derives values from the platform when possible).

### Cleanup

After inspecting the platform UI, run the final **Cleanup** cell to archive every resource created during this run.

As a fallback (e.g. if the kernel died mid-run), you can also run the standalone script which reads the same persisted state:

```bash
uv run python samples/resources/using-resources-clean.py
```


## Setup

Imports, helpers, and client connection (same as the script).

In [ ]:
from __future__ import annotations

import json
import os
import re
import tempfile
from datetime import datetime, timezone
from importlib.metadata import version as pkg_version
from pathlib import Path

import dotenv
from istari_digital_client import Configuration
from istari_digital_client.client import Client
from istari_digital_client.v3_client import V3Client
from istari_digital_client.v2.models.access_resource_type import AccessResourceType
from istari_digital_client.v2.models.permission import Permission
from istari_digital_client.v2.models.permission_resource_type import (
    PermissionResourceType,
)
from istari_digital_client.v2.models.permission_subject_type import (
    PermissionSubjectType,
)
from istari_digital_client.v3.models.archive_status import ArchiveStatus
from istari_digital_client.v3.models.resource_type_dto import ResourceTypeDto

EXPECTED_CLIENT_VERSION = "10.10.0"
RECIPE_TAG = "using-resources-recipe"

NOTEBOOK_DIR = Path.cwd()
SCRIPT_DIR = (
    NOTEBOOK_DIR
    if (NOTEBOOK_DIR / "using-resources.py").exists()
    else NOTEBOOK_DIR / "samples/resources"
)
SAMPLES_DIR = SCRIPT_DIR.parent
STATE_PATH = SCRIPT_DIR / ".using-resources-state.json"


def assert_client_version() -> None:
    installed = pkg_version("istari-digital-client")
    assert installed == EXPECTED_CLIENT_VERSION, (
        f"Expected istari-digital-client=={EXPECTED_CLIENT_VERSION}, got {installed}"
    )


def load_env() -> tuple[str, str]:
    dotenv.load_dotenv(SAMPLES_DIR / ".env")
    registry_url = os.environ.get("ISTARI_REGISTRY_URL")
    token = os.environ.get("ISTARI_PERSONAL_ACCESS_TOKEN")
    if not registry_url or not token:
        raise RuntimeError(
            "Set ISTARI_REGISTRY_URL and ISTARI_PERSONAL_ACCESS_TOKEN in samples/.env"
        )
    return registry_url, token


def ui_base_from_registry(registry_url: str) -> str:
    match = re.match(r"^(https?://)(?:fileservice-v2\.)?(.+?)/?$", registry_url)
    if not match:
        return registry_url.rstrip("/")
    return f"{match.group(1)}{match.group(2)}"


def save_state(state: dict) -> None:
    STATE_PATH.write_text(json.dumps(state, indent=2, default=str))
    print(f"Saved state → {STATE_PATH}")


def section(title: str) -> None:
    print(f"\n{'=' * 72}\n{title}\n{'=' * 72}")


def verify_resource_metadata(
    v3: V3Client,
    resource_id: str,
    *,
    name: str,
    display_name: str | None,
    external_identifier: str | None,
    version_name: str | None,
) -> None:
    fetched = v3.get_resource(resource_id=resource_id)
    assert fetched.name == name, f"name: got {fetched.name!r}, want {name!r}"
    assert fetched.display_name == display_name, (
        f"display_name: got {fetched.display_name!r}, want {display_name!r}"
    )
    assert fetched.external_identifier == external_identifier, (
        f"external_identifier: got {fetched.external_identifier!r}, "
        f"want {external_identifier!r}"
    )
    assert fetched.version_name == version_name, (
        f"version_name: got {fetched.version_name!r}, want {version_name!r}"
    )


def report_upload(
    ui_base: str,
    resource_id: str,
    *,
    name: str,
    display_name: str | None,
    external_identifier: str | None,
    version_name: str | None,
) -> None:
    print(f"Uploaded model resource_id={resource_id}")
    print(f"  name={name!r}")
    print(f"  display_name={display_name!r}")
    print(f"  external_identifier={external_identifier!r}")
    print(f"  version_name={version_name!r}")
    print("  assert: get_resource() metadata matches")
    print(f"UI: {ui_base}/files/{resource_id}")


assert_client_version()
registry_url, token = load_env()
config = Configuration(registry_url=registry_url, registry_auth_token=token)
v3 = V3Client(config)
client = Client(config)
ui_base = ui_base_from_registry(registry_url)

xlsx_v1 = SAMPLES_DIR / "Group3-UAS-Requirements.xlsx"
xlsx_v2 = SAMPLES_DIR / "Group3-UAS-Requirements-v2.xlsx"
if not xlsx_v1.is_file():
    raise FileNotFoundError(f"Sample spreadsheet not found: {xlsx_v1}")

run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
state: dict = {"run_id": run_id, "resource_ids": [], "comment_ids": []}

print(f"istari-digital-client {EXPECTED_CLIENT_VERSION}")
print(f"Registry: {registry_url}")


## 1 · Upload a resource from your device (no display_name)

UI: Files list — row shows the file name when `display_name` is omitted.

In [ ]:
section("1 · Upload a resource from your device (no display_name)")

# Standalone: this cell only needs the Setup cell — no upstream demo state.
state.setdefault("resource_ids", [])

minimal = v3.create_resource(
    path=xlsx_v1,
    resource_type=ResourceTypeDto.MODEL,
)
state["resource_ids"].append(minimal.resource_id)

assert minimal.resource_id
assert minimal.file_revision_id
assert not minimal.archived
verify_resource_metadata(
    v3,
    minimal.resource_id,
    name=xlsx_v1.name,
    display_name=None,
    external_identifier=None,
    version_name=None,
)
report_upload(
    ui_base,
    minimal.resource_id,
    name=xlsx_v1.name,
    display_name=None,
    external_identifier=None,
    version_name=None,
)

## 2 · Upload a resource with display_name

UI: Files list — row shows `display_name`; open detail page.

In [ ]:
section("2 · Upload a resource with display_name")

# Standalone: this cell only needs the Setup cell — no upstream demo state.
state.setdefault("resource_ids", [])

display_name = f"{RECIPE_TAG} UAS Requirements ({run_id})"
uploaded = v3.create_resource(
    path=xlsx_v1,
    resource_type=ResourceTypeDto.MODEL,
    display_name=display_name,
    description=f"Cookbook upload demo {run_id}",
    version_name="v1",
)
resource_id = uploaded.resource_id
state["primary_resource_id"] = resource_id
state["resource_ids"].append(resource_id)

assert uploaded.resource_id
assert uploaded.file_revision_id
assert not uploaded.archived
verify_resource_metadata(
    v3,
    resource_id,
    name=xlsx_v1.name,
    display_name=display_name,
    external_identifier=None,
    version_name="v1",
)
report_upload(
    ui_base,
    resource_id,
    name=xlsx_v1.name,
    display_name=display_name,
    external_identifier=None,
    version_name="v1",
)

## 3 · Upload with external id and version label

UI: File detail → External ID and Version fields on the revision.

In [ ]:
section("3 · Upload with external id and version label")

# Standalone: this cell only needs the Setup cell — no upstream demo state.
state.setdefault("resource_ids", [])

external_id = f"{RECIPE_TAG}-ext-{run_id}"
external_version = f"{RECIPE_TAG}-ver-{run_id}"
external_display_name = f"{RECIPE_TAG} external ids ({run_id})"
with_external = v3.create_resource(
    path=xlsx_v1,
    resource_type=ResourceTypeDto.MODEL,
    display_name=external_display_name,
    external_identifier=external_id,
    version_name=external_version,
)
external_resource_id = with_external.resource_id
state["external_resource_id"] = external_resource_id
state["resource_ids"].append(external_resource_id)

assert external_resource_id
assert with_external.file_revision_id
assert not with_external.archived
verify_resource_metadata(
    v3,
    external_resource_id,
    name=xlsx_v1.name,
    display_name=external_display_name,
    external_identifier=external_id,
    version_name=external_version,
)
report_upload(
    ui_base,
    external_resource_id,
    name=xlsx_v1.name,
    display_name=external_display_name,
    external_identifier=external_id,
    version_name=external_version,
)

## 4 · Search and filter resources

UI: Files list filters / search bar; file detail via direct lookup.

Also demonstrates `get_resource_revision`, lookup by `external_identifier`, and lookup by `external_identifier` + `version_name` (using the resource from step 3).

In [ ]:
section("4 · Search and filter resources")

# ── Standalone usage ─────────────────────────────────────────────────────────
# Run the Setup cell first. To run this cell on its own, set the inputs below
# (otherwise the values from the upstream demo cells are reused). Example:
#
#     resource_id = "<resource id with display_name+description+version_name>"
#     external_resource_id = "<resource id with an external_identifier set>"
#
# Optional: display_name, description_text, version_label, external_id,
# external_version. These are auto-derived from each resource if not set.
# ─────────────────────────────────────────────────────────────────────────────
if "resource_id" not in dir():
    resource_id = "REPLACE_WITH_RESOURCE_ID"
if "external_resource_id" not in dir():
    external_resource_id = resource_id  # default: search both against the same resource

detail = v3.get_resource(resource_id=resource_id)
external_detail = (
    detail if external_resource_id == resource_id
    else v3.get_resource(resource_id=external_resource_id)
)

# Derive expected values from each resource when not pre-set, so the strict
# assertions are meaningful in both flow and standalone mode.
if "display_name" not in dir():
    display_name = detail.display_name
if "description_text" not in dir():
    description_text = detail.description
if "version_label" not in dir():
    version_label = detail.version_name
if "external_id" not in dir():
    external_id = external_detail.external_identifier
if "external_version" not in dir():
    external_version = external_detail.version_name

# 1. Search by file name (may match many resources sharing the same filename).
by_name = v3.list_resources(name=[detail.name], size=50, include_total=True)
assert by_name.total is not None
assert by_name.total >= 1
name_match_ids = {r.resource_id for r in by_name.items}
assert resource_id in name_match_ids
print(f"name={detail.name!r} matched {by_name.total} resource(s)")

# 2. Get one resource by id.
assert detail.resource_id == resource_id
assert detail.display_name == display_name
assert detail.version_name == version_label
print(f"get_resource({resource_id}) → display_name={detail.display_name!r}")

# 3. Get a specific revision.
revision = v3.get_resource_revision(
    resource_id=resource_id,
    revision_id=detail.file_revision_id,
)
assert revision.file_revision_id == detail.file_revision_id
assert revision.owning_entity_id == resource_id
assert revision.version_name == version_label
print(
    f"get_resource_revision({resource_id}, {detail.file_revision_id}) "
    f"→ version_name={revision.version_name!r}"
)

# 4. Find resources by external_identifier (only if the external resource has one).
if external_id:
    by_external = v3.list_resources(external_identifier=[external_id])
    assert any(r.resource_id == external_resource_id for r in by_external.items)
    print(
        f"external_identifier={external_id!r} → resource_id={external_resource_id}"
    )

    if external_version:
        by_external_version = v3.list_resources(
            external_identifier=[external_id],
            version_name=[external_version],
        )
        match = next(
            (r for r in by_external_version.items if r.resource_id == external_resource_id),
            None,
        )
        assert match is not None
        assert match.external_identifier == external_id
        assert match.version_name == external_version
        print(
            f"external_identifier={external_id!r} version_name={external_version!r} "
            f"→ resource_id={match.resource_id}"
        )
else:
    print("(skip external_identifier lookup — external resource has no external_identifier)")

# 5. Owner filter — resolve current user first.
current_user = client.get_current_user()
assert current_user.id
print(
    f"get_current_user() → id={current_user.id!r} email={current_user.email!r}"
)

# 6. Other list_resources filters (each may match multiple rows).
if description_text:
    by_description = v3.list_resources(description=[description_text])
    assert any(r.resource_id == resource_id for r in by_description.items)

if version_label:
    by_version = v3.list_resources(version_name=[version_label])
    assert any(r.resource_id == resource_id for r in by_version.items)

by_owner = v3.list_resources(created_by_id=[detail.created_by_id])
assert any(r.resource_id == resource_id for r in by_owner.items)

by_status = v3.list_resources(archive_status=ArchiveStatus.ACTIVE)
assert any(r.resource_id == resource_id for r in by_status.items)
assert not detail.archived
print(
    f"Filter examples include resource_id={resource_id} "
    f"(owner={detail.created_by_id})"
)

## 5 · Browse resources you can access

UI: Files list page — all rows you have permission to view. Uses v3 cursor pagination.

In [ ]:
section("5 · Browse resources you can access")

# Standalone: this cell only needs the Setup cell. If `resource_id` is in scope
# (set by section 2 or by you), the cell also asserts that resource appears in
# the iteration; otherwise the assertion is skipped.

cursor = None
seen_ids: set[str] = set()
preview = 0
model_total: int | None = None
while True:
    page = v3.list_resources(
        type_name=["model"],
        archive_status=ArchiveStatus.ACTIVE,
        size=100,
        cursor=cursor,
        include_total=True,
    )
    if model_total is None:
        assert page.total is not None
        model_total = page.total
        print(f"Active models total: {model_total}")

    for r in page.items:
        seen_ids.add(r.resource_id)
        if preview < 5:
            print(f"  {r.display_name or r.name}  id={r.resource_id}")
            preview += 1

    cursor = page.next_page
    if not cursor:
        break

assert len(seen_ids) == model_total
if "resource_id" in dir():
    assert resource_id in seen_ids
print(f"Iterated all {len(seen_ids)} active model(s) across cursor pages")

## 6 · Add a comment

UI: File detail → Comments tab — attach a text note to the primary resource before inspecting its detail page.


In [ ]:
section("6 · Add a comment")

# ── Standalone usage ─────────────────────────────────────────────────────────
# Run the Setup cell first. To run this cell on its own, set:
#
#     resource_id = "<existing resource id>"
# ─────────────────────────────────────────────────────────────────────────────
if "resource_id" not in dir():
    resource_id = "REPLACE_WITH_RESOURCE_ID"
state.setdefault("comment_ids", [])

comment_body = Path(tempfile.gettempdir()) / f"{RECIPE_TAG}-comment-{run_id}.txt"
comment_body.write_text(f"Review note from {RECIPE_TAG} at {run_id}\n")
try:
    comment = v3.create_comment(resource_id=resource_id, path=comment_body)
    state["comment_ids"].append(comment.id)
    text = v3.get_content(comment).decode("utf-8")
    assert RECIPE_TAG in text
    print(f"Added comment id={comment.id}")
    print(f"Comments tab in UI: {ui_base}/files/{resource_id}")
finally:
    comment_body.unlink(missing_ok=True)

## 7 · View resource details

UI: File detail header, Versions tab, Comments tab, **Share** panel — includes the comment added in the previous step plus the list of users the resource is shared with and the role each one has.

This cell is **read-only**: it reports how the system is currently configured and does not modify the resource or its permissions.

### What `list_access` returns

`client.list_access` (`GET /api/v2/access/{resource_type}/{resource_id}`) returns the public `AccessRelationship` rows for the resource. Per the client:

- `AccessRelationship` is documented as *"PUBLIC: An access relationship that can be viewed/modified by a user of Istari."*
- The only `AccessSubjectType` exposed is `USER`, so every row is a single-user grant.
- The available `AccessRelation` values are `viewer`, `editor`, `owner`, `administrator`, `executor`, and `upstreamremoteowner`.
- The `resource_type` arg is derived from `detail.resource_type` so the same code works for `model`, `artifact`, or `file` resources.

If a caller can read the resource but no row for them appears in this list, the Python client doesn't expose any additional API to explain how their access was granted — so this notebook does not speculate. Inspect the platform UI's Share panel and any control-tag configuration for the resource if you need to investigate further.

### Control tags (`detail.control_tags`)

The cell also prints the resource's control tags. Per the `ControlTag` docstring: *"A control (essentially a tag) that is assigned to resource, files, and users. To have access to a resource or file that has one or more controls assigned, the user must have been assigned all the controls applied to the item."* — control tags are a **restriction** on top of the access grants above, not a grant in their own right.

### Effective permissions (`list_resource_type_permissions`)

The Share panel roles above are `AccessRelation` values (`viewer`, `editor`, `owner`, `administrator`). Separately, the authorization layer exposes action permissions via the `Permission` enum. For a model/artifact/file, the client and v3 API only document two that map to everyday UI capability:

| `Permission` | Where documented | UI meaning |
|---|---|---|
| `view` | v3 `list_resources` / `get_resource` responses refer to VIEW permission; `test_access.py` uses `Permission.VIEW` with `list_resource_type_permissions` | Can open and read the resource |
| `edit` | Same endpoint; no dedicated test for models, but this is the edit counterpart to `view` | Can modify the resource |

Other `Permission` values in the enum (`manage`, `access`, `archive`, `execute`, `access_view_manage`, `access_edit_manage`, `access_administrate_manage`, …) are not referenced by the resources API or resource tests in istari-python-client 10.10.0, so this notebook does not query them.

`client.list_resource_type_permissions(subject_type, subject_id, resource_type, permission)` (`GET /api/v2/access/{subject_type}/{subject_id}/{resource_type}?permission=…`) returns every resource of `resource_type` on which the subject holds the requested permission. The cell checks whether `resource_id` appears in those rows — surfacing effective access even when the caller has no row in `list_access`. One HTTP call per permission; failures are reported and skipped so one bad call does not abort the cell.


In [ ]:
section("7 · View resource details")

# ── Standalone usage ─────────────────────────────────────────────────────────
# Run the Setup cell first. To run this cell on its own, set:
#
#     resource_id = "<existing resource id>"
#
# `detail` is always re-fetched; `current_user` / `comment` are reused if
# already in scope.
# ─────────────────────────────────────────────────────────────────────────────
if "resource_id" not in dir():
    resource_id = "REPLACE_WITH_RESOURCE_ID"


def describe_access_on_resource(
    client,
    *,
    user_id: str,
    resource_id: str,
    resource_type: PermissionResourceType,
    explicit_access_rows: list,
) -> dict[str, str | bool | None]:
    """Explicit Share role (AccessRelation) + effective view/edit (Permission).

    Share-panel roles come from list_access (viewer/editor/owner/administrator).
    Effective capability comes from list_resource_type_permissions with
    Permission.VIEW and Permission.EDIT — the only Permission values referenced
    for resource access in the v3 API docs and istari-python-client tests.
    Returns None for effective_view/effective_edit when the lookup errors.
    """
    explicit = next(
        (a.relation.value for a in explicit_access_rows if a.subject_id == user_id),
        None,
    )

    def has_effective(perm: Permission) -> bool | None:
        try:
            rows = client.list_resource_type_permissions(
                subject_type=PermissionSubjectType.USER,
                subject_id=user_id,
                resource_type=resource_type,
                permission=perm,
            )
        except Exception:
            return None
        return any(r.resource_id == resource_id for r in rows)

    return {
        "explicit_share_role": explicit,
        "effective_view": has_effective(Permission.VIEW),
        "effective_edit": has_effective(Permission.EDIT),
    }


detail = v3.get_resource(resource_id=resource_id)
if "current_user" not in dir():
    current_user = client.get_current_user()

revisions = v3.list_resource_revisions(resource_id=resource_id, include_total=True)
comments_page = v3.list_comments(resource_id=resource_id, include_total=True)

assert detail.file_id
assert detail.file_revision_id
assert revisions.total is not None and revisions.total >= 1
assert comments_page.total is not None and comments_page.total >= 1
if "comment" in dir():
    assert any(c.id == comment.id for c in comments_page.items)
print(f"resource_id={detail.resource_id}")
print(f"file_id={detail.file_id}  revision_id={detail.file_revision_id}")
print(f"Revisions: {revisions.total}  Comments: {comments_page.total}")

# Control tags on the resource. Per the ControlTag docstring, having a tag is
# a *restriction* (a user must hold all of the tags assigned to the resource
# to access it), not a grant of access in itself.
if detail.control_tags:
    print(
        "Control tags: "
        + ", ".join(t.name for t in detail.control_tags if t and t.name)
    )
else:
    print("Control tags: (none)")

# Public access relationships for the resource. Returns user-subject rows as
# documented by AccessRelationship ("PUBLIC: An access relationship that can
# be viewed/modified by a user of Istari"). Read-only here.
#
# `client.list_access` is on the v2 API and its signature requires an
# `AccessResourceType` (v2 enum: model, artifact, job, file, filerevision,
# tool, function, ...). `detail.resource_type` is a `ResourceTypeDto` from
# the v3 resources API (model, artifact, file only) — a different Python
# type with the same wire string values for the three categories that exist
# in v3, so we re-wrap the string to satisfy the v2 endpoint's type.
access_resource_type = AccessResourceType(detail.resource_type.value)
access_list = client.list_access(
    resource_type=access_resource_type,
    resource_id=resource_id,
)

print(f"Access relationships ({len(access_list)} row(s)):")
for entry in access_list:
    info = entry.subject_info
    if info and (info.username or info.email):
        who = info.username or info.email
    else:
        who = entry.subject_id
    cross_tenant = " [cross-tenant]" if info and info.cross_tenant_user else ""
    self_marker = " (you)" if entry.subject_id == current_user.id else ""
    print(
        f"  - {entry.relation.value:<14} "
        f"{entry.subject_type.value}={who}{self_marker}{cross_tenant}"
    )

caller_label = current_user.email or current_user.id
access_summary = describe_access_on_resource(
    client,
    user_id=current_user.id,
    resource_id=resource_id,
    resource_type=PermissionResourceType(detail.resource_type.value),
    explicit_access_rows=access_list,
)

print(f"Access summary for {caller_label}:")
if access_summary["explicit_share_role"]:
    print(f"  explicit Share role: {access_summary['explicit_share_role']!r}")
else:
    print("  explicit Share role: (none)")

for perm_label, key in [("view", "effective_view"), ("edit", "effective_edit")]:
    value = access_summary[key]
    if value is None:
        print(f"  effective {perm_label:<4} error")
    else:
        print(f"  effective {perm_label:<4} {'YES' if value else 'no'}")

## 8 · Upload a new version

UI: Versions tab → new revision row.

In [ ]:
section("8 · Upload a new version")

# ── Standalone usage ─────────────────────────────────────────────────────────
# Run the Setup cell first. To run this cell on its own, set:
#
#     resource_id = "<existing resource id>"
#
# `display_name` defaults to the resource's current display_name if not set.
# ─────────────────────────────────────────────────────────────────────────────
if "resource_id" not in dir():
    resource_id = "REPLACE_WITH_RESOURCE_ID"

detail = v3.get_resource(resource_id=resource_id)
if "display_name" not in dir():
    display_name = detail.display_name or detail.name

version_path = xlsx_v2 if xlsx_v2.is_file() else xlsx_v1
revision_v2 = v3.create_resource_revision(
    resource_id=resource_id,
    path=version_path,
    description=f"Second revision {run_id}",
    version_name="v2",
    display_name=f"{display_name} (v2)",
)
state["revision_v2_id"] = revision_v2.file_revision_id

assert revision_v2.owning_entity_id == resource_id
assert revision_v2.file_revision_id != detail.file_revision_id
rev_list = v3.list_resource_revisions(resource_id=resource_id, include_total=True)
assert rev_list.total is not None and rev_list.total >= 2
print(f"New revision_id={revision_v2.file_revision_id}")

## 9 · Compare resource versions

UI: Versions tab → select two revisions → Compare. SDK compares content hashes (no dedicated compare API in 10.10.0).

In [ ]:
section("9 · Compare resource versions")

# ── Standalone usage ─────────────────────────────────────────────────────────
# Run the Setup cell first. To run this cell on its own, set:
#
#     resource_id = "<existing resource id with 2+ revisions>"
#
# Optional: rev_v1_id / rev_v2_id (specific revisions to compare). When not
# set, the cell compares the oldest and newest revisions for the resource.
# ─────────────────────────────────────────────────────────────────────────────
if "resource_id" not in dir():
    resource_id = "REPLACE_WITH_RESOURCE_ID"

if "rev_v1_id" not in dir():
    rev_v1_id = detail.file_revision_id if "detail" in dir() else None
if "rev_v2_id" not in dir():
    rev_v2_id = revision_v2.file_revision_id if "revision_v2" in dir() else None

if not rev_v1_id or not rev_v2_id:
    rev_page = v3.list_resource_revisions(resource_id=resource_id, include_total=True)
    assert rev_page.total is not None and rev_page.total >= 2, (
        "Resource needs at least 2 revisions to compare — run section 8 first "
        "or pick a different resource_id."
    )
    revs_sorted = sorted(rev_page.items, key=lambda r: r.created)
    rev_v1_id = rev_v1_id or revs_sorted[0].file_revision_id
    rev_v2_id = rev_v2_id or revs_sorted[-1].file_revision_id

rev_v1 = v3.get_resource_revision(resource_id=resource_id, revision_id=rev_v1_id)
rev_v2 = v3.get_resource_revision(resource_id=resource_id, revision_id=rev_v2_id)
content_v1 = v3.get_content(rev_v1)
content_v2 = v3.get_content(rev_v2)
same_bytes = content_v1 == content_v2
print(f"v1 bytes={len(content_v1)}  v2 bytes={len(content_v2)}  identical={same_bytes}")
if not same_bytes:
    assert rev_v1.content_token is not None and rev_v2.content_token is not None
    assert rev_v1.content_token.sha != rev_v2.content_token.sha
print(f"Compare in UI: {ui_base}/files/{resource_id} (Versions → Compare)")

## 10 · Download resource content

UI: Download button on file detail / revision.

In [ ]:
section("10 · Download resource content")

# ── Standalone usage ─────────────────────────────────────────────────────────
# Run the Setup cell first. To run this cell on its own, set:
#
#     resource_id = "<existing resource id>"
#
# The download size assertion checks against the source upload only when
# `xlsx_v1` is known to be the original — i.e. when the upload happened in this
# kernel. Otherwise we just verify the file was written non-empty.
# ─────────────────────────────────────────────────────────────────────────────
if "resource_id" not in dir():
    resource_id = "REPLACE_WITH_RESOURCE_ID"

detail = v3.get_resource(resource_id=resource_id)

download_dir = SCRIPT_DIR / "downloads"
download_dir.mkdir(exist_ok=True)
suffix = Path(detail.name or "").suffix or ".bin"
dest = download_dir / f"{RECIPE_TAG}-{resource_id}-{run_id}{suffix}"
dest.write_bytes(v3.get_content(detail))
if detail.size is not None:
    assert dest.stat().st_size == detail.size, (
        f"downloaded {dest.stat().st_size} bytes, expected {detail.size}"
    )
else:
    assert dest.stat().st_size > 0
state["download_path"] = str(dest)
print(f"Wrote {dest} ({dest.stat().st_size} bytes)")

## 11 · Archive a resource (throwaway)

Primary demo resource stays active for UI review.

In [ ]:
section("11 · Archive a resource (throwaway)")

# Standalone: this cell only needs the Setup cell. It uploads a fresh resource
# and then archives it, so it never depends on upstream demo state.
state.setdefault("resource_ids", [])

archive_display_name = f"{RECIPE_TAG} archive demo ({run_id})"
archive_target = v3.create_resource(
    path=xlsx_v1,
    resource_type=ResourceTypeDto.FILE,
    display_name=archive_display_name,
)
state["archive_demo_resource_id"] = archive_target.resource_id
state["resource_ids"].append(archive_target.resource_id)

verify_resource_metadata(
    v3,
    archive_target.resource_id,
    name=xlsx_v1.name,
    display_name=archive_display_name,
    external_identifier=None,
    version_name=None,
)

v3.archive_resource(resource_id=archive_target.resource_id)
archived = v3.get_resource(resource_id=archive_target.resource_id)
assert archived.archived
archived_list = v3.list_resources(
    resource_id=[archive_target.resource_id],
    archive_status=ArchiveStatus.ARCHIVED,
)
assert len(archived_list.items) == 1
print(f"Archived throwaway resource_id={archive_target.resource_id}")

## 12 · Cleanup — archive demo resources

Run the next cell after you're done inspecting the UI. It uses the in-memory `state` dict (the same one populated throughout this notebook) to archive every resource created during this run and remove the local download.

This is the inline equivalent of [`using-resources-clean.py`](./using-resources-clean.py), which you can still run separately if you skip this cell — `save_state(state)` below persists the same ids to `.using-resources-state.json` as a fallback.

In [ ]:
section("12 · Cleanup — archive demo resources")

# ── Standalone usage ─────────────────────────────────────────────────────────
# Run the Setup cell first. To run cleanup on its own (e.g. after a kernel
# restart), the cell loads the persisted state file written by `save_state`.
# You can also build `state` by hand, for example:
#
#     state = {"resource_ids": ["<id-1>", "<id-2>"], "download_path": None}
# ─────────────────────────────────────────────────────────────────────────────
if "state" not in dir() or not state.get("resource_ids"):
    if STATE_PATH.is_file():
        state = json.loads(STATE_PATH.read_text())
        print(f"Loaded state from {STATE_PATH}")
    else:
        state = {"resource_ids": [], "comment_ids": []}

save_state(state)

cleanup_resource_ids = list(dict.fromkeys(state.get("resource_ids", [])))
cleanup_download_path = state.get("download_path")

print(
    f"Cleaning run_id={state.get('run_id', '?')} "
    f"({len(cleanup_resource_ids)} resource(s))"
)

for rid in cleanup_resource_ids:
    try:
        current = v3.get_resource(resource_id=rid)
    except Exception as exc:
        print(f"  skip resource_id={rid}: {exc}")
        continue

    if current.archived:
        print(f"  already archived: {rid}")
        continue

    v3.archive_resource(resource_id=rid)
    archived_check = v3.get_resource(resource_id=rid)
    assert archived_check.archived
    still_active = v3.list_resources(
        resource_id=[rid],
        archive_status=ArchiveStatus.ACTIVE,
    )
    assert len(still_active.items) == 0
    print(f"  archived: {rid}")

if cleanup_download_path:
    download_file = Path(cleanup_download_path)
    if download_file.is_file():
        download_file.unlink()
        print(f"  removed download: {download_file}")

if STATE_PATH.is_file():
    STATE_PATH.unlink()
    print(f"  removed state file: {STATE_PATH}")

print("Cleanup complete — you can rerun this notebook from the top.")

## Done

All demo resources for this run have been archived and the local download removed.